In [2]:
#YOLO26s

In [1]:
from ultralytics import YOLO

# 1. Load a pretrained model (YOLOv8 Nano is great for small objects like checkboxes)
model = YOLO("yolo26l.pt")
model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=True, track_

In [2]:
"""
Converts your JSON ground-truth labels → YOLO .txt format.

JSON input per file:
  { "checkboxes": [[cx,cy], ...],       ← class 0
    "lines":      [[x1,y1,x2,y2], ...], ← class 1
    "boxes":      [[x1,y1,x2,y2], ...]} ← class 2

YOLO output per line:
  <class_id> <cx_norm> <cy_norm> <w_norm> <h_norm>
"""

import os, json
from PIL import Image

# ── CONFIG ──────────────────────────────────────────
GT_FOLDER       = "clicking_mechanism/annotations"
IMAGE_FOLDER    = "data"
YOLO_LABELS_DIR = "clicking_mechanism/yolo_labels_v11" 
LINE_PADDING    = 0.01   # 1% padding on each side of a line bbox
CHECKBOX_SIZE   = 0.02   # checkboxes get a 2% fixed-size bbox
# ────────────────────────────────────────────────────

def point_to_yolo(cx, cy, w, h):
    return cx/w, cy/h, CHECKBOX_SIZE, CHECKBOX_SIZE

def line_to_yolo(x1, y1, x2, y2, w, h):
    xmin = max(0,  min(x1,x2) - LINE_PADDING * w)
    xmax = min(w,  max(x1,x2) + LINE_PADDING * w)
    ymin = max(0,  min(y1,y2) - LINE_PADDING * h)
    ymax = min(h,  max(y1,y2) + LINE_PADDING * h)
    return (xmin+xmax)/(2*w), (ymin+ymax)/(2*h), (xmax-xmin)/w, (ymax-ymin)/h

def box_to_yolo(x1, y1, x2, y2, w, h):
    return (x1+x2)/(2*w), (y1+y2)/(2*h), abs(x2-x1)/w, abs(y2-y1)/h

def convert():
    os.makedirs(YOLO_LABELS_DIR, exist_ok=True)
    # Only process files present in BOTH folders
    gt_files    = set(f for f in os.listdir(GT_FOLDER)    if f.endswith(".json"))
    img_files   = set(f.replace(".png",".json") for f in os.listdir(IMAGE_FOLDER) if f.endswith(".png"))
    files       = gt_files & img_files
    skipped     = gt_files - img_files

    for fname in skipped:
        print(f"[SKIP] No image for {fname}")

    for fname in files:
        img_w, img_h = Image.open(os.path.join(IMAGE_FOLDER, fname.replace(".json",".png"))).size
        data = json.load(open(os.path.join(GT_FOLDER, fname)))
        rows = []

        for cx, cy           in data.get("checkboxes", []):
            rows.append("0 {:.6f} {:.6f} {:.6f} {:.6f}".format(*point_to_yolo(cx, cy, img_w, img_h)))
        for x1,y1,x2,y2      in data.get("lines", []):
            rows.append("1 {:.6f} {:.6f} {:.6f} {:.6f}".format(*line_to_yolo(x1,y1,x2,y2,img_w,img_h)))
        for x1,y1,x2,y2      in data.get("boxes", []):
            rows.append("2 {:.6f} {:.6f} {:.6f} {:.6f}".format(*box_to_yolo(x1,y1,x2,y2,img_w,img_h)))

        with open(os.path.join(YOLO_LABELS_DIR, fname.replace(".json",".txt")), "w") as f:
            f.write("\n".join(rows))

    print(f"\n✅ Converted {len(files)} files | Skipped {len(skipped)} (no image)")

if __name__ == "__main__":
    convert()


✅ Converted 408 files | Skipped 0 (no image)


In [3]:
"""
Splits data into train/val, writes dataset.yaml, and trains YOLOv8.
Run AFTER step1_convert_labels.py.
"""

import os, shutil, random, yaml
from ultralytics import YOLO

IMAGE_FOLDER    = "data"
YOLO_LABELS_DIR = "clicking_mechanism/yolo_labels_v11" 
# ── CONFIG ──────────────────────────────────────────
DATASET_DIR     = "dataset_v26_l"          # will be created
MODEL_SIZE      = "yolo26l.pt"       # n/s/m/l/x — s is a good default
EPOCHS          = 100
IMGSZ           = 640
BATCH           = 16
VAL_SPLIT       = 0.2
SEED            = 42
# ────────────────────────────────────────────────────

CLASSES = ["checkbox", "line", "box"]

def prepare_dataset():
    for split in ("train", "val"):
        os.makedirs(f"{DATASET_DIR}/images/{split}", exist_ok=True)
        os.makedirs(f"{DATASET_DIR}/labels/{split}", exist_ok=True)

    # Only include images that have a matching label file
    label_stems = set(f[:-4] for f in os.listdir(YOLO_LABELS_DIR) if f.endswith(".txt"))
    image_files = [f for f in os.listdir(IMAGE_FOLDER)
                   if f.endswith(".png") and f[:-4] in label_stems]

    random.seed(SEED)
    random.shuffle(image_files)
    split_idx  = int(len(image_files) * (1 - VAL_SPLIT))
    train_imgs = image_files[:split_idx]
    val_imgs   = image_files[split_idx:]

    for split, imgs in [("train", train_imgs), ("val", val_imgs)]:
        for img_fname in imgs:
            stem = img_fname[:-4]
            shutil.copy(os.path.join(IMAGE_FOLDER,    img_fname),
                        f"{DATASET_DIR}/images/{split}/{img_fname}")
            shutil.copy(os.path.join(YOLO_LABELS_DIR, stem + ".txt"),
                        f"{DATASET_DIR}/labels/{split}/{stem}.txt")

    print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)}")

    # Write dataset.yaml
    yaml_path = os.path.join(DATASET_DIR, "dataset.yaml")
    yaml_data = {
        "path": os.path.abspath(DATASET_DIR),
        "train": "images/train",
        "val":   "images/val",
        "nc":    len(CLASSES),
        "names": CLASSES,
    }
    with open(yaml_path, "w") as f:
        yaml.dump(yaml_data, f, default_flow_style=False)
    print(f"Dataset YAML written to {yaml_path}")
    return yaml_path

def train(yaml_path):
    model = YOLO(MODEL_SIZE)
    model.train(
        data    = yaml_path,
        epochs  = EPOCHS,
        imgsz   = IMGSZ,
        batch   = BATCH,
        project = "run_v26l/runs/train",
        name    = "form_detector",
    )
    print("\n✅ Training complete. Best weights: runs/train/form_detector/weights/best.pt")

if __name__ == "__main__":
    yaml_path = prepare_dataset()
    train(yaml_path)

Train: 326 | Val: 82
Dataset YAML written to dataset_v26_l/dataset.yaml
New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.7 🚀 Python-3.12.5 torch-2.9.1 CPU (Apple M3 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_v26_l/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26l.pt, momentum=0.937, mosaic=1.0

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile


      1/100         0G      3.869      4.932    0.01711        970        640: 0% ──────────── 0/21  1:11

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile


      1/100         0G      3.846      4.927    0.01799        844        640: 5% ╸─────────── 1/21 232.7s/it 2:21<1:17:35

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 33% ━━━━──────── 1/3 73.8s/it 22.1s<2:28

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng war

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 67% ━━━━━━━━──── 2/3 36.8s/it 39.1s<36.8s

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 16.6s/it 49.7s
                   all         82       4149      0.641      0.617      0.588      0.321
              checkbox         69       2887      0.524      0.459      0.357      0.111
                  line         56        612      0.671      0.638      0.649      0.361
                   box         54        650      0.727      0.755      0.759      0.492
Speed: 0.6ms preprocess, 581.3ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /Users/gauurab/Documents/Projects/salary-nku/camp-norse/Detect_Regions_in_PDF/runs/detect/run_v26l/runs/train/form_detector

✅ Training complete. Best weights: runs/train/form_detector/weights/best.pt


In [4]:
"""
Runs YOLOv8 inference on test images and computes
Precision / Recall / F1 / Accuracy (Jaccard) per class,
matching your original tolerance-based evaluation logic.
"""

import os, json, math
from PIL import Image
from ultralytics import YOLO

# ── CONFIG ──────────────────────────────────────────
WEIGHTS_PATH  = "runs/detect/run_v26l/runs/train/form_detector/weights/best.pt"
GT_FOLDER       = "clicking_mechanism/annotations"
IMAGE_FOLDER    = "data"
YOLO_LABELS_DIR = "clicking_mechanism/yolo_labels_v11" 
TOLERANCES    = [0.05, 0.10, 0.20]
CONF_THRESH   = 0.25
CLASS_NAMES   = {0: "Checkboxes", 1: "Lines", 2: "Boxes"}
# ────────────────────────────────────────────────────

def load_gt(path, img_w, img_h):
    """Load GT JSON and convert everything to pixel (x1,y1,x2,y2) boxes."""
    data = json.load(open(path))
    gt = {0: [], 1: [], 2: []}

    for cx, cy in data.get("checkboxes", []):
        s = 0.01 * min(img_w, img_h)
        gt[0].append((cx-s, cy-s, cx+s, cy+s))

    for x1,y1,x2,y2 in data.get("lines", []):
        pad_x = 0.01 * img_w
        pad_y = 0.01 * img_h
        gt[1].append((min(x1,x2)-pad_x, min(y1,y2)-pad_y,
                      max(x1,x2)+pad_x, max(y1,y2)+pad_y))

    for x1,y1,x2,y2 in data.get("boxes", []):
        gt[2].append((x1, y1, x2, y2))

    return gt

def match_boxes(gt_boxes, pred_boxes, tol_x, tol_y):
    """Same order-independent closest-first matching as your original code."""
    candidates = []
    for gi, (gx1,gy1,gx2,gy2) in enumerate(gt_boxes):
        for pi, (px1,py1,px2,py2) in enumerate(pred_boxes):
            if (abs(px1-gx1)<=tol_x and abs(py1-gy1)<=tol_y and
                abs(px2-gx2)<=tol_x and abs(py2-gy2)<=tol_y):
                dist = math.sqrt((gx1-px1)**2+(gy1-py1)**2) + \
                       math.sqrt((gx2-px2)**2+(gy2-py2)**2)
                candidates.append((dist, gi, pi))

    candidates.sort(key=lambda x: x[0])
    matched_gt, matched_pred, count = set(), set(), 0
    for _, gi, pi in candidates:
        if gi not in matched_gt and pi not in matched_pred:
            matched_gt.add(gi); matched_pred.add(pi); count += 1
    return count

def compute_metrics(matches, total_pred, total_gt):
    p   = matches / total_pred if total_pred else 0
    r   = matches / total_gt   if total_gt   else 0
    f1  = (2*p*r / (p+r))      if (p+r)      else 0
    den = total_gt + total_pred - matches
    acc = matches / den         if den        else 0
    return p, r, f1, acc

def yolo_box_to_pixels(box, img_w, img_h):
    """Convert YOLO result xyxy (already in pixels) to tuple."""
    x1, y1, x2, y2 = box
    return float(x1), float(y1), float(x2), float(y2)

def evaluate():
    model = YOLO(WEIGHTS_PATH)

    # Verified file pairs only
    gt_stems  = set(f[:-5] for f in os.listdir(GT_FOLDER)    if f.endswith(".json"))
    img_stems = set(f[:-4]  for f in os.listdir(IMAGE_FOLDER) if f.endswith(".png"))
    stems     = sorted(gt_stems & img_stems)

    if not stems:
        print("No matching GT+image pairs found. Check your paths.")
        return

    for tol in TOLERANCES:
        totals = {cls: [0,0,0] for cls in CLASS_NAMES}  # [gt, pred, match]

        print("\n" + "="*60)
        print(f"TOLERANCE: {int(tol*100)}%")
        print("="*60)

        for stem in stems:
            img_path = os.path.join(IMAGE_FOLDER, stem + ".png")
            gt_path  = os.path.join(GT_FOLDER,    stem + ".json")
            img_w, img_h = Image.open(img_path).size
            tol_x, tol_y = img_w * tol, img_h * tol

            gt = load_gt(gt_path, img_w, img_h)

            results = model.predict(img_path, conf=CONF_THRESH, verbose=False)[0]

            # Group predictions by class
            pred = {0: [], 1: [], 2: []}
            for box in results.boxes:
                cls_id = int(box.cls)
                if cls_id in pred:
                    pred[cls_id].append(yolo_box_to_pixels(box.xyxy[0], img_w, img_h))

            for cls_id in CLASS_NAMES:
                matches = match_boxes(gt[cls_id], pred[cls_id], tol_x, tol_y)
                totals[cls_id][0] += len(gt[cls_id])
                totals[cls_id][1] += len(pred[cls_id])
                totals[cls_id][2] += matches

        for cls_id, name in CLASS_NAMES.items():
            gt_n, pred_n, match_n = totals[cls_id]
            p, r, f1, acc = compute_metrics(match_n, pred_n, gt_n)
            print(f"\n{name}:")
            print(f"  Precision: {p:.3f} | Recall: {r:.3f} | F1: {f1:.3f} | Accuracy (Jaccard): {acc:.3f}")

if __name__ == "__main__":
    evaluate()


TOLERANCE: 5%


libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: 


Checkboxes:
  Precision: 0.958 | Recall: 0.562 | F1: 0.708 | Accuracy (Jaccard): 0.549

Lines:
  Precision: 0.790 | Recall: 0.784 | F1: 0.787 | Accuracy (Jaccard): 0.649

Boxes:
  Precision: 0.624 | Recall: 0.634 | F1: 0.629 | Accuracy (Jaccard): 0.459

TOLERANCE: 10%


libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: 


Checkboxes:
  Precision: 0.973 | Recall: 0.571 | F1: 0.719 | Accuracy (Jaccard): 0.562

Lines:
  Precision: 0.806 | Recall: 0.799 | F1: 0.802 | Accuracy (Jaccard): 0.670

Boxes:
  Precision: 0.663 | Recall: 0.673 | F1: 0.668 | Accuracy (Jaccard): 0.502

TOLERANCE: 20%


libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: 


Checkboxes:
  Precision: 0.981 | Recall: 0.575 | F1: 0.725 | Accuracy (Jaccard): 0.569

Lines:
  Precision: 0.825 | Recall: 0.818 | F1: 0.821 | Accuracy (Jaccard): 0.697

Boxes:
  Precision: 0.761 | Recall: 0.773 | F1: 0.767 | Accuracy (Jaccard): 0.622


In [5]:
"""
Predict on a single random form image and visualize results.
"""

import os, random
from PIL import Image, ImageDraw
from ultralytics import YOLO

# ── CONFIG ──────────────────────────────────────────
WEIGHTS_PATH  = "runs/detect/run_v26l/runs/train/form_detector/weights/best.pt"
IMAGE_FOLDER  = "data"
OUTPUT_PATH   = "prediction_sample.png"
CONF_THRESH   = 0.25
# ────────────────────────────────────────────────────

CLASS_NAMES  = {0: "checkbox", 1: "line", 2: "box"}
CLASS_COLORS = {0: "blue",     1: "green", 2: "red"}

def predict_random():
    images = [f for f in os.listdir(IMAGE_FOLDER) if f.endswith(".png")]
    chosen = random.choice(images)
    img_path = os.path.join(IMAGE_FOLDER, chosen)
    print(f"Running prediction on: {chosen}")

    model   = YOLO(WEIGHTS_PATH)
    results = model.predict(img_path, conf=CONF_THRESH, verbose=False)[0]

    img  = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)

    counts = {0: 0, 1: 0, 2: 0}
    for box in results.boxes:
        cls_id = int(box.cls)
        conf   = float(box.conf)
        x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]
        color  = CLASS_COLORS[cls_id]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        draw.text((x1, max(0, y1 - 12)), f"{CLASS_NAMES[cls_id]} {conf:.2f}", fill=color)
        counts[cls_id] += 1

    img.save(OUTPUT_PATH)

    print(f"\nDetected:")
    for cls_id, name in CLASS_NAMES.items():
        print(f"  {name}: {counts[cls_id]}")
    print(f"\nSaved to: {OUTPUT_PATH}")

if __name__ == "__main__":
    predict_random()

Running prediction on: form457.png

Detected:
  checkbox: 26
  line: 15
  box: 8

Saved to: prediction_sample.png


In [9]:
"""
Predict on a single specified form image and visualize results.
"""

import os
from PIL import Image, ImageDraw
from ultralytics import YOLO

# ── CONFIG ──────────────────────────────────────────
WEIGHTS_PATH  = "runs/detect/run_v11/runs/train/form_detector/weights/best.pt"
IMAGE_PATH    = "form-test.jpg"   # <-- directly specify your test image
OUTPUT_PATH   = "prediction_sample.png"
CONF_THRESH   = 0.25
# ────────────────────────────────────────────────────

CLASS_NAMES  = {0: "checkbox", 1: "line", 2: "box"}
CLASS_COLORS = {0: "blue",     1: "green", 2: "red"}

def predict_image():
    print(f"Running prediction on: {IMAGE_PATH}")

    model   = YOLO(WEIGHTS_PATH)
    results = model.predict(IMAGE_PATH, conf=CONF_THRESH, verbose=False)[0]

    img  = Image.open(IMAGE_PATH).convert("RGB")
    draw = ImageDraw.Draw(img)

    counts = {0: 0, 1: 0, 2: 0}

    for box in results.boxes:
        cls_id = int(box.cls)
        conf   = float(box.conf)
        x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]

        color = CLASS_COLORS.get(cls_id, "yellow")
        label = CLASS_NAMES.get(cls_id, "unknown")

        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        draw.text((x1, max(0, y1 - 12)), f"{label} {conf:.2f}", fill=color)

        if cls_id in counts:
            counts[cls_id] += 1

    img.save(OUTPUT_PATH)

    print("\nDetected:")
    for cls_id, name in CLASS_NAMES.items():
        print(f"  {name}: {counts[cls_id]}")

    print(f"\nSaved to: {OUTPUT_PATH}")

if __name__ == "__main__":
    predict_image()

Running prediction on: form-test.jpg

Detected:
  checkbox: 0
  line: 13
  box: 4

Saved to: prediction_sample.png
